In [289]:
import numpy as np
S0 = 100
L = 80
J = 24
r = 0.05
sigma = 0.2
K= 110
T = 2
M = 100000

In [ ]:
def generate_brownian(n):
    G = np.random.normal(0, 1, (n, J)) # (n, J)
    step = T/J
    increment = G * np.sqrt(step) # (n, J)
    return np.concatenate((np.zeros((n,1)), increment.cumsum(axis=1)), axis=1) # (n, J+1)

def simulate_trajectory(W):
    t = np.linspace(0,T, J+1) # (J+1,)
    return S0 * np.exp((r-sigma*sigma/2)*t + sigma * W)

In [ ]:
def call_barrier_monitoring(W): # f
    trajectory = simulate_trajectory(W) # (n, J+1)
    payoff = np.maximum(trajectory[:,-1] - K, 0.0) # (n,)
    return payoff * np.all(trajectory>=L, axis=1) * np.exp(-r * T) # (n,)

In [292]:
n = 1000
W = generate_brownian(n)
x = W[:,-1]
f = call_barrier_monitoring(W)

In [293]:
def u(lam):
    expo = np.exp(-lam*W[:,-1])
    f_sq = f*f
    return lam*lam*T/2 + np.log((expo@f_sq)/n)

In [294]:
def u_p(lam):
    num = np.exp(-lam*x) * f*f 
    weights = num/ num.sum()
    return lam*T - weights @ x

In [295]:
def u_pp(lam):
    num = np.exp(-lam*x) * f*f 
    weights = num/ num.sum()
    return T + weights@(x*x) -(weights @ x)**2

In [296]:
def lam(p):
    lam = 0.0
    for _ in range(p):
        lam -= u_p(lam)/u_pp(lam)
    return lam
lam_star = lam(5)

In [304]:
def IC_MC():
    W_new = generate_brownian(M)
    t = np.linspace(0, T, J+1)
    W_shifted = (W_new + lam_star * t)
    f_values = call_barrier_monitoring(W_shifted)
    Y = f_values * np.exp(-lam_star*W_new[:,-1] - lam_star*lam_star*T/2)
    return np.mean(Y), 1.96 * np.std(Y)/np.sqrt(M)
IC_MC()

(np.float64(11.247115902710098), np.float64(0.03701374818534504))

In [302]:
def plain_MC():
    W_new = generate_brownian(M)
    X = call_barrier_monitoring(W_new)
    return np.mean(X), 1.96 * np.std(X)/np.sqrt(M)
plain_MC()

(np.float64(11.271745851989845), np.float64(0.12227281585131007))